# 21 — Modèle final

## 1. Sources et variables fixes

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
TABLES = ROOT / "data" / "02_Tables_originales"
MANIFESTES = ROOT / "data" / "05_Manifestes"

HORIZON, FENETRE = 730, (640, 820)
TOLERANCE_SCORE = 90          # jours AVANT la visite : on prend la mesure la plus recente, jamais une mesure posterieure

def lire(prefixe):
    f = sorted(TABLES.rglob(prefixe + "*.csv"))
    if not f:
        return None
    t = pd.read_csv(f[-1], dtype="string", encoding="utf-8-sig")
    t.columns = t.columns.str.strip().str.upper()
    if "PTID" in t.columns:
        t["PTID"] = t["PTID"].str.strip().str.upper()
    return t

def en_date(s):
    return pd.to_datetime(s.astype("string").str.strip().str.slice(0, 10),
                          format="%Y-%m-%d", errors="coerce").dt.normalize()

# --- diagnostics propres : memes regles que le notebook 18 ---
dxb = lire("All_Subjects_DXSUM_")
if dxb is None:
    raise FileNotFoundError("Table DXSUM introuvable.")
dxb["date_dx"] = en_date(dxb["EXAMDATE"])
dxb["code_dx"] = pd.to_numeric(dxb["DIAGNOSIS"], errors="coerce")
dxb = dxb.loc[dxb["PTID"].str.fullmatch(r"\d{3}_S_\d{4,}", na=False)
              & dxb["date_dx"].notna() & dxb["code_dx"].isin([1, 2, 3]),
              ["PTID", "date_dx", "code_dx"]]
n_codes = dxb.groupby(["PTID", "date_dx"])["code_dx"].nunique()
conflits = n_codes[n_codes > 1].reset_index()[["PTID", "date_dx"]]
dxb = (dxb.loc[~pd.MultiIndex.from_frame(dxb[["PTID", "date_dx"]])
               .isin(pd.MultiIndex.from_frame(conflits))]
       .drop_duplicates().sort_values(["PTID", "date_dx"]))
dxb["code_dx"] = dxb["code_dx"].astype(int)
dxb["rang"] = dxb.groupby("PTID").cumcount()
print("Visites diagnostiques retenues :", len(dxb),
      "| participants :", dxb["PTID"].nunique())

# --- variables fixes, lues directement dans les tables sources ---
fixes = pd.DataFrame({"PTID": sorted(dxb["PTID"].unique())})

demo = lire("All_Subjects_PTDEMOG_")
if demo is not None:
    d = demo.copy()
    for brut, nom in [("PTGENDER", "sexe"), ("PTEDUCAT", "education")]:
        if brut in d.columns:
            d[nom] = pd.to_numeric(d[brut], errors="coerce")
    col_dob = next((c for c in ("PTDOBYY", "PTDOB", "PTDOBYEAR") if c in d.columns), None)
    if col_dob:
        d["naissance"] = pd.to_numeric(
            d[col_dob].astype("string").str.extract(r"(\d{4})")[0], errors="coerce")
        d.loc[~d["naissance"].between(1850, 2026), "naissance"] = np.nan
    garde = [c for c in ("sexe", "education", "naissance") if c in d.columns]
    d = (d[["PTID"] + garde].dropna(how="all", subset=garde)
         .drop_duplicates("PTID", keep="first"))
    fixes = fixes.merge(d, on="PTID", how="left", validate="one_to_one")

apoe = lire("All_Subjects_APOERES_")
if apoe is not None and "GENOTYPE" in apoe.columns:
    g = apoe[["PTID", "GENOTYPE"]].copy()
    geno = g["GENOTYPE"].astype("string").str.replace(" ", "", regex=False)
    g["APOE_e4"] = geno.where(geno.str.fullmatch(r"[234]/[234]", na=False)).map(
        lambda s: str(s).count("4") if pd.notna(s) else np.nan)
    g = g.dropna(subset=["APOE_e4"]).drop_duplicates("PTID", keep="first")[["PTID", "APOE_e4"]]
    fixes = fixes.merge(g, on="PTID", how="left", validate="one_to_one")

COLS_FIXES = [c for c in ("sexe", "education", "APOE_e4") if c in fixes.columns]
print("\nVariables fixes :", COLS_FIXES + (["age"] if "naissance" in fixes.columns else []))
display((fixes[[c for c in COLS_FIXES + ["naissance"] if c in fixes.columns]]
         .notna().mean() * 100).round(1).to_frame("couverture_%"))

## 2. Construire la cohorte pour un nombre de visites donné

Pour un K donné : on prend les **K premières visites** diagnostiques du participant,
on pose **Tk** = date de la K-ième, et la cible est le diagnostic à la visite la plus
proche de Tk + 730 jours dans la fenêtre [640, 820].

Rien de ce qui suit Tk n'entre dans les variables.

In [ ]:
SCORES = {"All_Subjects_MMSE_": ("VISDATE", {"MMSCORE": "MMSE"}),
          "All_Subjects_CDR_": ("VISDATE", {"CDGLOBAL": "CDR_global", "CDRSB": "CDR_SB"}),
          "All_Subjects_ADAS_": ("VISDATE", {"TOTAL13": "ADAS13"}),
          "All_Subjects_FAQ_": ("VISDATE", {"FAQTOTAL": "FAQ"})}

_cache = {}
def table_score(prefixe, col_date, corr):
    if prefixe in _cache:
        return _cache[prefixe]
    t = lire(prefixe)
    if t is None or col_date not in t.columns:
        _cache[prefixe] = None
        return None
    t["date_mesure"] = en_date(t[col_date])
    noms = []
    for brut, nom in corr.items():
        if brut in t.columns:
            t[nom] = pd.to_numeric(t[brut], errors="coerce")
            t[nom] = t[nom].where(t[nom] >= 0)   # -1 = valeur manquante dans ADNI, pas un vrai score
            noms.append(nom)
    t = (t.loc[t["date_mesure"].notna(), ["PTID", "date_mesure"] + noms]
         .dropna(subset=noms, how="all")
         .drop_duplicates(subset=["PTID", "date_mesure"])
         .sort_values("date_mesure")) if noms else None
    _cache[prefixe] = t
    return t

def construire(K):
    """Retourne (ptids, cube (n,K,v), fixes (n,f), y, noms_variables_visite)."""
    nb = dxb.groupby("PTID")["rang"].max() + 1
    d = dxb.loc[dxb["PTID"].isin(nb[nb >= K + 1].index)]

    ancre = (d.loc[d["rang"] == K - 1].set_index("PTID")[["date_dx", "code_dx"]]
             .rename(columns={"date_dx": "date_Tk", "code_dx": "dx_Tk"}))
    suite = d.merge(ancre, on="PTID")
    suite["delai"] = (suite["date_dx"] - suite["date_Tk"]).dt.days
    cand = suite.loc[suite["delai"].between(*FENETRE)].copy()
    cand["distance"] = (cand["delai"] - HORIZON).abs()
    cible = (cand.sort_values(["PTID", "distance", "date_dx"])
             .drop_duplicates("PTID")[["PTID", "code_dx"]]
             .rename(columns={"code_dx": "dx_cible"}))

    base = ancre.reset_index().merge(cible, on="PTID", how="inner")
    base = base.loc[base["dx_Tk"].isin([1, 2])].copy()
    base["cible"] = (base["dx_cible"] > base["dx_Tk"]).astype(int)
    if base.empty:
        return None

    panel = (d.loc[d["rang"] < K, ["PTID", "date_dx", "code_dx", "rang"]]
             .loc[lambda t: t["PTID"].isin(set(base["PTID"]))]
             .sort_values(["PTID", "rang"]).copy())
    for prefixe, (cd, corr) in SCORES.items():
        t = table_score(prefixe, cd, corr)
        if t is None:
            continue
        noms = [c for c in t.columns if c not in ("PTID", "date_mesure")]
        lien = pd.merge_asof(panel.sort_values("date_dx"),
                             t.loc[t["PTID"].isin(set(panel["PTID"]))],
                             by="PTID", left_on="date_dx", right_on="date_mesure",
                             direction="backward",
                             tolerance=pd.Timedelta(days=TOLERANCE_SCORE))
        panel = panel.merge(lien[["PTID", "rang"] + noms], on=["PTID", "rang"], how="left")

    var_visite = ["code_dx"] + [c for c in
                                ["MMSE", "CDR_global", "CDR_SB", "ADAS13", "FAQ"]
                                if c in panel.columns]
    ptids = sorted(base["PTID"])
    cube = (panel.set_index(["PTID", "rang"])[var_visite]
            .reindex(pd.MultiIndex.from_product([ptids, range(K)], names=["PTID", "rang"]))
            .to_numpy(dtype=float).reshape(len(ptids), K, len(var_visite)))

    f = base.set_index("PTID").loc[ptids, ["date_Tk"]].join(
        fixes.set_index("PTID"), how="left")
    if "naissance" in f.columns:
        f["age"] = f["date_Tk"].dt.year - f["naissance"]
        f.loc[~f["age"].between(40, 120), "age"] = np.nan
    cols_f = COLS_FIXES + (["age"] if "age" in f.columns else [])
    F = f[cols_f].to_numpy(dtype=float)
    y = base.set_index("PTID").loc[ptids, "cible"].to_numpy(dtype=int)
    return ptids, cube, F, y, var_visite, cols_f

COHORTES = {}
lignes = []
for K in (2, 3, 4):
    r = construire(K)
    if r is None:
        continue
    COHORTES[K] = r
    ptids, cube, F, y, vv, cf = r
    lignes.append({"K visites": K, "participants": len(y), "aggravations": int(y.sum()),
                   "taux": round(float(y.mean()), 3),
                   "couverture variables fixes_%": round(100 * (1 - np.isnan(F).mean()), 1)})
display(pd.DataFrame(lignes))
print("Variables par visite :", COHORTES[max(COHORTES)][4])
print("Variables fixes      :", COHORTES[max(COHORTES)][5])

## 3. Mettre le jeu de test de côté — maintenant

L'affectation se fait **par participant** et de façon déterministe : un participant
tiré au test l'est pour tous les K. Les cohortes K = 2, 3 et 4 se recouvrent largement,
donc sans cette précaution un patient pourrait servir au développement pour un K et au
test pour un autre — et le test ne serait plus indépendant.

À partir d'ici, et jusqu'à la section 6, **le jeu de test n'est plus touché**.

In [ ]:
PART_TEST = 0.25
GRAINE = 7

tous = sorted(set().union(*[set(COHORTES[K][0]) for K in COHORTES]))
rng = np.random.default_rng(GRAINE)
melange = np.array(tous)[rng.permutation(len(tous))]
PTID_TEST = set(melange[:int(round(PART_TEST * len(tous)))])
print(f"{len(tous)} participants au total -> {len(PTID_TEST)} reserves au test\n")

def separer(K):
    ptids, cube, F, y, vv, cf = COHORTES[K]
    m_test = np.array([p in PTID_TEST for p in ptids])
    return m_test

verif = []
for K in COHORTES:
    ptids, cube, F, y, vv, cf = COHORTES[K]
    m = separer(K)
    verif.append({"K": K,
                  "developpement": int((~m).sum()), "positifs_dev": int(y[~m].sum()),
                  "test": int(m.sum()), "positifs_test": int(y[m].sum()),
                  "taux_dev": round(float(y[~m].mean()), 3),
                  "taux_test": round(float(y[m].mean()), 3)})
display(pd.DataFrame(verif))
print("Les taux d'aggravation doivent etre proches entre developpement et test.")

## 4. Phase 1 — choisir K et le modèle, sur le développement uniquement

Pour chaque K, deux représentations :

- **statique** : la dernière visite seule, plus les variables fixes ;
- **longitudinal** : les K visites mises bout à bout, plus les pentes entre la première
  et la dernière, plus les variables fixes.

Validation croisée 5 plis, à l'intérieur du développement. Le jeu de test reste fermé.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def tuyau_logreg():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("ech", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])

def tuyau_foret():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("clf", RandomForestClassifier(n_estimators=300, random_state=0,
                                                    class_weight="balanced", n_jobs=-1))])

def probas(Xa, ya, Xt):
    pl = tuyau_logreg().fit(Xa, ya).predict_proba(Xt)[:, 1]
    pf = tuyau_foret().fit(Xa, ya).predict_proba(Xt)[:, 1]
    return {"Regression logistique": pl, "Random forest": pf, "Ensemble": (pl + pf) / 2}

def representations(K):
    ptids, cube, F, y, vv, cf = COHORTES[K]
    n = len(ptids)
    stat = np.hstack([cube[:, -1, :], F])
    pentes = cube[:, -1, :] - cube[:, 0, :]
    longi = np.hstack([cube.reshape(n, -1), pentes, F])
    return {"statique": stat, "longitudinal": longi}

resultats = []
for K in sorted(COHORTES):
    m_test = separer(K)
    y = COHORTES[K][3]
    for forme, Xtot in representations(K).items():
        Xd, yd = Xtot[~m_test], y[~m_test]
        auc = {n: [] for n in ("Regression logistique", "Random forest", "Ensemble")}
        for app, val in CV.split(Xd, yd):
            p = probas(Xd[app], yd[app], Xd[val])
            for n in auc:
                auc[n].append(roc_auc_score(yd[val], p[n]))
        for n, v in auc.items():
            resultats.append({"K": K, "forme": forme, "modele": n,
                              "roc_auc": float(np.mean(v)), "ecart_type": float(np.std(v)),
                              "n_dev": int(len(yd))})

grille = pd.DataFrame(resultats).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(grille.round(3).head(12))

MEILLEUR = grille.iloc[0]
# Regle de parcimonie (dite "d'une erreur-type") : parmi les configurations dont la note est a moins
# d'un ecart-type de la meilleure, on retient la plus SIMPLE (statique avant longitudinal, moins de
# visites, regression logistique avant random forest avant ensemble). Un ecart plus petit que le bruit
# entre plis n'est pas une amelioration demontree, et un modele plus simple se deploie plus facilement.
ORDRE_MODELE = {"Regression logistique": 0, "Random forest": 1, "Ensemble": 2}
proches = grille.loc[grille["roc_auc"] >= MEILLEUR["roc_auc"] - MEILLEUR["ecart_type"]].copy()
proches["complexite"] = list(zip(proches["forme"].map({"statique": 0, "longitudinal": 1}),
                                 proches["K"], proches["modele"].map(ORDRE_MODELE)))
CHOIX = proches.sort_values("complexite").iloc[0]
print(f"Meilleure note brute : K={int(MEILLEUR['K'])} {MEILLEUR['forme']} {MEILLEUR['modele']} "
      f"= {MEILLEUR['roc_auc']:.3f} +/- {MEILLEUR['ecart_type']:.3f}")
print(f"{len(proches)} configuration(s) a moins d'un ecart-type -> on retient la plus simple.")
K_FINAL, FORME_FINALE, MODELE_FINAL = int(CHOIX["K"]), CHOIX["forme"], CHOIX["modele"]
print(f"\n>>> Choix retenu : K = {K_FINAL} visites, representation « {FORME_FINALE} », "
      f"modele « {MODELE_FINAL} »")
print(f"    Note en developpement : {CHOIX['roc_auc']:.3f} +/- {CHOIX['ecart_type']:.3f} "
      f"(sur {int(CHOIX['n_dev'])} participants)")
print("    Cette note est optimiste : elle a servi a faire le choix. "
      "La vraie note est en section 6.")

# apport du temps, a K choisi
pivot = (grille.loc[grille["modele"] == MODELE_FINAL]
         .pivot(index="K", columns="forme", values="roc_auc").round(3))
pivot["apport du temps"] = (pivot["longitudinal"] - pivot["statique"]).round(3)
print("\nApport du suivi dans le temps, par nombre de visites :")
display(pivot)

## 5. Phase 2 — choisir le seuil, toujours sur le développement

Le modèle donne une probabilité ; le seuil décide à partir de quelle valeur on alerte.
Il se choisit **sur le développement**, jamais sur le test — sinon le test ne mesurerait
plus rien.

On prend le seuil de Youden, celui qui équilibre le mieux sensibilité et spécificité.

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, balanced_accuracy_score

m_test = separer(K_FINAL)
y_tot = COHORTES[K_FINAL][3]
X_tot = representations(K_FINAL)[FORME_FINALE]
X_dev, y_dev = X_tot[~m_test], y_tot[~m_test]
X_test, y_test = X_tot[m_test], y_tot[m_test]
print(f"Developpement : {len(y_dev)} participants ({y_dev.mean():.1%} d'aggravations)")
print(f"Test          : {len(y_test)} participants ({y_test.mean():.1%} d'aggravations)")

# probabilites hors echantillon sur le developpement, pour choisir le seuil
oof = np.full(len(y_dev), np.nan)
for app, val in CV.split(X_dev, y_dev):
    oof[val] = probas(X_dev[app], y_dev[app], X_dev[val])[MODELE_FINAL]

fpr_d, tpr_d, seuils_d = roc_curve(y_dev, oof)
SEUIL = float(seuils_d[int(np.argmax(tpr_d - fpr_d))])
print(f"\nSeuil de Youden choisi sur le developpement : {SEUIL:.3f}")

def mesures(yv, p, s):
    pred = (p >= s).astype(int)
    vn, fp, fn, vp = confusion_matrix(yv, pred, labels=[0, 1]).ravel()
    return {"seuil": round(s, 3),
            "sensibilite": round(vp / max(vp + fn, 1), 3),
            "specificite": round(vn / max(vn + fp, 1), 3),
            "VPP": round(vp / max(vp + fp, 1), 3),
            "VPN": round(vn / max(vn + fn, 1), 3),
            "exactitude equilibree": round(balanced_accuracy_score(yv, pred), 3)}

print("Au developpement, a ce seuil :")
display(pd.DataFrame([mesures(y_dev, oof, SEUIL)]))

## 6. Phase 3 — le test final, une seule fois

On entraîne le modèle retenu sur **tout** le développement, puis on prédit le jeu de test,
qui n'a jamais été vu. **Ce chiffre est le seul qu'on puisse annoncer.**

L'intervalle de confiance est calculé par rééchantillonnage : le test ne compte que
quelques dizaines d'aggravations, la note est donc entourée d'incertitude. L'annoncer
sans intervalle serait trompeur.

In [ ]:
p_test = probas(X_dev, y_dev, X_test)[MODELE_FINAL]
auc_test = roc_auc_score(y_test, p_test)

# intervalle de confiance a 95 % par reechantillonnage
rngb = np.random.default_rng(0)
tirages = []
for _ in range(2000):
    i = rngb.integers(0, len(y_test), len(y_test))
    if len(np.unique(y_test[i])) < 2:
        continue
    tirages.append(roc_auc_score(y_test[i], p_test[i]))
bas, haut = np.percentile(tirages, [2.5, 97.5])

print("=" * 72)
print(f">>> NOTE FINALE SUR LE JEU DE TEST INDEPENDANT")
print(f"    ROC-AUC = {auc_test:.3f}   (IC 95 % : {bas:.3f} – {haut:.3f})")
print(f"    {len(y_test)} participants, dont {int(y_test.sum())} aggravations")
print("=" * 72)
print(f"\nPour memoire, note en developpement : {CHOIX['roc_auc']:.3f} "
      f"(optimiste, elle a servi au choix)")

mes_test = mesures(y_test, p_test, SEUIL)
print(f"\nAu seuil {SEUIL:.3f}, choisi sans regarder le test :")
display(pd.DataFrame([mes_test]))

pred = (p_test >= SEUIL).astype(int)
mat = pd.DataFrame(confusion_matrix(y_test, pred, labels=[0, 1]),
                   index=["reel : ne s'aggrave pas", "reel : s'aggrave"],
                   columns=["predit : ne s'aggrave pas", "predit : s'aggrave"])
display(mat)
vp = int(mat.iloc[1, 1]); fn = int(mat.iloc[1, 0]); fp = int(mat.iloc[0, 1])
print(f"Sur {int(y_test.sum())} patients qui se sont vraiment aggraves, "
      f"le modele en detecte {vp} et en manque {fn}, au prix de {fp} fausses alertes.")

## 7. La figure du modèle final

In [ ]:
from sklearn.calibration import calibration_curve

fpr, tpr, _ = roc_curve(y_test, p_test)
i_seuil = int(np.argmin(np.abs(np.sort(p_test)[::-1] - SEUIL))) if len(p_test) else 0

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

sel = grille.loc[grille["modele"] == MODELE_FINAL].sort_values(["forme", "K"])
etiq = [f"K={int(r.K)} {r.forme[:5]}." for r in sel.itertuples()]
pos = np.arange(len(sel))
a1.barh(pos, sel["roc_auc"], xerr=sel["ecart_type"], capsize=3,
        color=["#4C72B0" if (r.K == K_FINAL and r.forme == FORME_FINALE) else "#B0B0B0"
               for r in sel.itertuples()])
a1.set_yticks(pos); a1.set_yticklabels(etiq, fontsize=8)
a1.axvline(0.5, color="grey", ls="--", lw=1)
a1.set_xlim(0.45, 1.0); a1.set_xlabel("ROC-AUC (developpement)")
a1.set_title("Choix du nombre de visites", fontsize=10)

a2.plot(fpr, tpr, color="#4C72B0", lw=2, label=f"AUC = {auc_test:.3f}")
a2.plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="hasard")
a2.scatter([1 - mes_test["specificite"]], [mes_test["sensibilite"]],
           color="#C44E52", s=70, zorder=5)
a2.annotate(f"seuil {SEUIL:.2f}", (1 - mes_test["specificite"], mes_test["sensibilite"]),
            textcoords="offset points", xytext=(8, -12), fontsize=9)
a2.set_xlabel("1 - specificite"); a2.set_ylabel("sensibilite")
a2.set_title("Courbe ROC — jeu de test independant", fontsize=10)
a2.legend(loc="lower right")

nb_groupes = 4 if len(y_test) < 250 else 5
frac, moy = calibration_curve(y_test, p_test, n_bins=nb_groupes, strategy="quantile")
a3.plot(moy, frac, marker="o", color="#4C72B0")
a3.plot([0, 1], [0, 1], color="grey", ls="--", lw=1)
a3.set_xlabel("probabilite predite"); a3.set_ylabel("frequence observee")
a3.set_title(f"Calibration ({nb_groupes} groupes)", fontsize=10)
plt.show()

## 8. Export

In [ ]:
SORTIE = MANIFESTES / ("modele_final_21_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
SORTIE.mkdir(parents=True, exist_ok=False)

grille.round(4).to_csv(SORTIE / "choix_K_et_modele.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(verif).to_csv(SORTIE / "separation_dev_test.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([mes_test]).to_csv(SORTIE / "mesures_test.csv", index=False, encoding="utf-8-sig")
mat.reset_index().rename(columns={"index": "reel"}).to_csv(
    SORTIE / "matrice_confusion_test.csv", index=False, encoding="utf-8-sig")

final = {
    "experience": "21_modele_final_longitudinal",
    "date": datetime.now().isoformat(),
    "choix": {"K_visites": K_FINAL, "representation": FORME_FINALE, "modele": MODELE_FINAL},
    "developpement": {"participants": int(len(y_dev)),
                      "aggravations": int(y_dev.sum()),
                      "roc_auc_optimiste": round(float(CHOIX["roc_auc"]), 3)},
    "test_independant": {"participants": int(len(y_test)),
                         "aggravations": int(y_test.sum()),
                         "roc_auc": round(float(auc_test), 3),
                         "IC95": [round(float(bas), 3), round(float(haut), 3)],
                         "seuil": round(SEUIL, 3),
                         **{k: v for k, v in mes_test.items() if k != "seuil"}},
    "cible": "diagnostic a environ 2 ans superieur a celui de la K-ieme visite",
    "variables_par_visite": COHORTES[K_FINAL][4],
    "variables_fixes": COHORTES[K_FINAL][5],
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
}
(SORTIE / "modele_final.json").write_text(json.dumps(final, ensure_ascii=False, indent=2),
                                          encoding="utf-8")
print(json.dumps(final["test_independant"], ensure_ascii=False, indent=2))
print("\nResultats enregistres dans :", SORTIE)